# ACDC initial exploration and basic cleaning
Inspect the original data, identify issues and apply basic corrections.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)

In [ ]:
df_raw = pd.read_csv("../data/raw/acdc_dataset.csv", sep=";")
df = df_raw.copy()
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## Missing values and duplicates

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
pd.Series({
    "Exact duplicate rows": df.duplicated().sum(),
    "Repeated member IDs": df["member_id"].duplicated().sum()
})

## Categories

In [ ]:
df[["gender", "country", "industry", "plan_tier"]].nunique()

In [ ]:
df["plan_tier"].value_counts(dropna=False)

In [ ]:
df["gender"].value_counts(dropna=False)

In [ ]:
df["country"].value_counts(dropna=False).head(15)

In [ ]:
df["industry"].value_counts(dropna=False).head(15)

## Values to review
The age limits below flag records for review. They are not removal rules.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
negatives = (df[numeric_columns] < 0).sum()
negatives[negatives > 0]

In [ ]:
pd.Series({
    "Age below 18 or above 100": ((df["age"] < 18) | (df["age"] > 100)).sum(),
    "Plan with zero days per week": (df["plan_days_per_week"] == 0).sum(),
    "Plan with more than seven days per week": (df["plan_days_per_week"] > 7).sum(),
    "Zero entitled visits": (df["entitled_visits_last_period"] == 0).sum(),
    "Actual visits above entitlement": (df["actual_visits_last_period"] > df["entitled_visits_last_period"]).sum()
})

## Revenue format
A failed conversion does not mean that the original value is missing.

In [ ]:
revenue_numeric = pd.to_numeric(df["lifetime_value"], errors="coerce")
conversion_issues = df["lifetime_value"].notna() & revenue_numeric.isna()

print("Nonmissing values that need format conversion:", conversion_issues.sum())
print("Negative revenue values already readable:", (revenue_numeric < 0).sum())
df.loc[conversion_issues, ["lifetime_value"]].head(8)

## Dates
Parse both observed date formats in a separate table.

In [ ]:
dates = pd.DataFrame(index=df.index)

for column in ["membership_start", "membership_end", "last_badge_swipe"]:
    iso_dates = pd.to_datetime(df[column], format="%Y-%m-%d", errors="coerce")
    other_dates = pd.to_datetime(df[column], format="%d/%m/%Y", errors="coerce")
    dates[column] = iso_dates.fillna(other_dates)

dates.isna().sum()

In [ ]:
dates.agg(["min", "max"])

In [ ]:
end_1970 = dates["membership_end"] == pd.Timestamp("1970-01-01")

pd.Series({
    "End date equal to 1970-01-01": end_1970.sum(),
    "End before start, excluding 1970": ((dates["membership_end"] < dates["membership_start"]) & ~end_1970).sum(),
    "Last swipe before start": (dates["last_badge_swipe"] < dates["membership_start"]).sum(),
    "Last swipe after first end, excluding 1970": ((dates["last_badge_swipe"] > dates["membership_end"]) & ~end_1970).sum()
})

## Initial plots of the original data

In [ ]:
df[["age", "total_visits", "avg_weekly_visits"]].hist(
    bins=30, figsize=(12, 3), layout=(1, 3)
)
plt.tight_layout()
plt.show()

In [ ]:
df["plan_tier"].value_counts().sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("Number of members")
plt.ylabel("Plan before cleaning")
plt.tight_layout()
plt.show()

## Basic cleaning
All changes below affect `df`. The original `df_raw` is preserved.

### Exact duplicates
Remove rows only when every column is identical. Then check that member IDs are unique.

In [ ]:
df = df_raw.copy()
rows_before = len(df)
df = df.drop_duplicates().copy()
missing_before_cleaning = df.isna().sum()

print("Rows removed:", rows_before - len(df))
print("Rows remaining:", len(df))
print("Repeated member IDs:", df["member_id"].duplicated().sum())

### Category labels
Remove surrounding spaces, use lowercase and merge clear aliases. Missing values and other labels are preserved.

In [ ]:
category_columns = ["gender", "country", "industry", "plan_tier"]
categories_before = df[category_columns].nunique()

for column in category_columns:
    df[column] = df[column].str.strip().str.lower()

In [ ]:
df["gender"] = df["gender"].replace({
    "f": "female", "m": "male", "o": "other"
})

df["plan_tier"] = df["plan_tier"].replace({
    "hot desk": "hot_desk", "hotdesk": "hot_desk",
    "full time": "full_time", "fulltime": "full_time",
    "dedicated desk": "dedicated_desk", "dedicateddesk": "dedicated_desk",
    "private office": "private_office", "privateoffice": "private_office"
})

In [ ]:
df["country"] = df["country"].replace({
    "pt": "portugal", "portucal": "portugal",
    "uk": "united kingdom", "gb": "united kingdom",
    "u.k.": "united kingdom",
    "de": "germany", "deutschland": "germany",
    "es": "spain", "espana": "spain", "españa": "spain",
    "fr": "france", "ie": "ireland",
    "nl": "netherlands", "the netherlands": "netherlands",
    "holland": "netherlands"
})

In [ ]:
df["industry"] = df["industry"].replace({
    "it": "information technology",
    "design/creative": "design & creative",
    "design and creative": "design & creative",
    "marketing/pr": "marketing & pr",
    "marketing and pr": "marketing & pr",
    "realestate": "real estate", "real-estate": "real estate"
})

We keep `tech` separate because it is broader than information technology. We also keep `unknown`, `other` and `other (europe)` as distinct labels.

In [ ]:
pd.DataFrame({
    "Categories before": categories_before,
    "Categories after": df[category_columns].nunique()
})

In [ ]:
df["plan_tier"].value_counts(dropna=False)

### Revenue conversion
In this file, commas separate thousands and dots mark decimals. Remove euro symbols, commas and surrounding spaces, then convert to numbers. Negative values remain for later review.

In [ ]:
revenue_text = df["lifetime_value"].str.replace("€", "", regex=False)
revenue_text = revenue_text.str.replace(",", "", regex=False).str.strip()
df["lifetime_value"] = pd.to_numeric(revenue_text, errors="raise")

print("Missing revenue values:", df["lifetime_value"].isna().sum())
print("Negative revenue values:", (df["lifetime_value"] < 0).sum())
print("New missing revenue values:",
      df["lifetime_value"].isna().sum() - missing_before_cleaning["lifetime_value"])

### Date conversion and interpretation
Both date formats were parsed above. Store the parsed dates in the working table.

The value `1970-01-01` looks like a placeholder, but its meaning is not documented. It does not prove that a membership is active. Keep it flagged and avoid using it to calculate duration. The guidelines define `membership_end` as the first end date, so later visits may follow a renewal.

In [ ]:
date_columns = ["membership_start", "membership_end", "last_badge_swipe"]

for column in date_columns:
    df[column] = dates[column]

df[date_columns].dtypes

In [ ]:
end_1970 = df["membership_end"] == pd.Timestamp("1970-01-01")
end_before_start = (df["membership_end"] < df["membership_start"]) & ~end_1970
swipe_before_start = df["last_badge_swipe"] < df["membership_start"]

pd.Series({
    "End date equal to 1970-01-01": end_1970.sum(),
    "End before start, excluding 1970": end_before_start.sum(),
    "Last swipe before start": swipe_before_start.sum()
})

In [ ]:
recency_to_end = (df["membership_end"] - df["last_badge_swipe"]).dt.days
print("Recency values matching membership end:",
      (df["days_since_last_visit"] == recency_to_end).sum())

recency_reference = df["last_badge_swipe"] + pd.to_timedelta(
    df["days_since_last_visit"], unit="D"
)
recency_reference.value_counts().head()

The implied reference date varies across members. Recency often matches the membership end rather than a single extraction date. Its definition needs confirmation before recalculating it.

### Service flags
The original `uses_print` values are only 0 and 1. Convert them to match the other service flags.

In [ ]:
df["uses_print"] = df["uses_print"].astype(bool)
df.filter(like="uses_").dtypes

### Check the result
Only exact duplicate rows were removed. The conversions should preserve missing values and all remaining records.

In [ ]:
pd.Series({
    "Rows remaining": len(df),
    "Exact duplicate rows": df.duplicated().sum(),
    "Repeated member IDs": df["member_id"].duplicated().sum(),
    "Missing counts unchanged after conversions": df.isna().sum().equals(missing_before_cleaning)
})

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
negatives = (df[numeric_columns] < 0).sum()
negatives[negatives > 0]

## Agreed data quality treatments
Keep all remaining members. Store the basic cleaned data before applying the agreed rules.

In [ ]:
df_basic = df.copy()
guest_missing_no_use = df["guest_passes_issued"].isna() & ~df["uses_guest_passes"]
meeting_missing_no_use = df["meeting_room_hours"].isna() & ~df["uses_meeting_rooms"]

### Age
Keep age 14. For this project, treat ages below 10 or above 120 as invalid. Fill missing and invalid ages with the median of the remaining ages. This is an estimate, so record how many values were filled.

In [ ]:
age_invalid = (df["age"] < 10) | (df["age"] > 120)
df["age"] = df["age"].mask(age_invalid)
age_imputed = df["age"].isna()
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

print("Invalid ages:", age_invalid.sum())
print("Median age:", age_median)
print("Ages filled:", age_imputed.sum())
print("Members aged 14 retained:", (df["age"] == 14).sum())

### Unknown categories
Use `unknown` for missing gender, country and plan labels. Keep these members and their other information. These fields can describe clusters later; this does not decide which features will enter the clustering model.

In [ ]:
description_columns = ["gender", "country", "plan_tier"]

for column in description_columns:
    df[column] = df[column].fillna("unknown")

(df[description_columns] == "unknown").sum()

### Invalid counts and hours
Replace negative counts and hours with missing values. A plan cannot provide more than seven days per week, so mark those values as missing too. Keep the members. Negative revenue and recency remain under review because they need separate interpretation.

In [ ]:
negative_columns = [
    "n_colleagues_on_account", "total_visits", "meeting_room_hours",
    "event_space_hours", "guest_passes_issued", "n_renewals"
]

for column in negative_columns:
    df[column] = df[column].mask(df[column] < 0)

invalid_plan_days = df["plan_days_per_week"] > 7
df["plan_days_per_week"] = df["plan_days_per_week"].mask(invalid_plan_days)

print("Invalid weekly plan days marked missing:", invalid_plan_days.sum())

### Missing service quantities
Assume zero when the original quantity is missing and the usage flag is False. This is an imputation assumption, especially for meeting rooms where some flags conflict with recorded hours.

Keep missing quantities for True flags as NaN. True means the member used the service; it does not tell us the number of passes or hours. Preserve recorded positive hours and the original flags in the conflicting records.

In [ ]:
df.loc[guest_missing_no_use, "guest_passes_issued"] = 0
df.loc[meeting_missing_no_use, "meeting_room_hours"] = 0

print("Missing guest pass quantities filled with zero:", guest_missing_no_use.sum())
print("Missing meeting hours filled with zero:", meeting_missing_no_use.sum())
df[["guest_passes_issued", "meeting_room_hours"]].isna().sum()

### Unresolved data quality flags
The flags below refer to values before these treatments. Small visit differences remain unchanged and documented. Their cause is unknown. Date conflicts remain unresolved, and future duration features must exclude affected intervals.

A zero revenue value together with recency 999 suggests a possible data quality pattern. It does not establish that the member is inactive or that either value is missing. Keep both values unchanged.

In [ ]:
visit_gap = df_basic["actual_visits_last_period"] - df_basic["total_visits"]
meeting_conflict = (~df_basic["uses_meeting_rooms"]) & (df_basic["meeting_room_hours"] > 0)

quality_flags = pd.DataFrame({
    "First end date unresolved (1970)": end_1970,
    "First end before start": end_before_start,
    "Last swipe before start": swipe_before_start,
    "Visits exceed total by one or two": visit_gap.between(1, 2),
    "Visits exceed total by more than two": visit_gap > 2,
    "Positive meeting hours with unused flag": meeting_conflict,
    "Zero revenue and recency 999": (df_basic["lifetime_value"] == 0) & (df_basic["days_since_last_visit"] == 999)
})
quality_flags.sum()

### Check the treatments
Confirm that all members remain and inspect the missing values still requiring decisions. Quality flags and imputation masks are kept separately from the dataset.

In [ ]:
print("Members before treatments:", len(df_basic))
print("Members after treatments:", len(df))
missing_after_treatments = df.isna().sum()
missing_after_treatments[missing_after_treatments > 0]

## Next steps
Decide how to handle the remaining missing quantities, unresolved dates, negative revenue, recency and extreme values. Continue exploring behaviour and value before selecting clustering features. No activity status has been inferred.